In [1]:
import math 

In [4]:
def sigmoid(x):
    x= max(-500,min(500,x))
    return 1/(1+math.exp(-x))

def sigmoid_derivative(x):
    return sigmoid(x)*(1-sigmoid(x))

def tanh(x):
    return math.tanh(x)

def tanh_derivative(x):
    t= tanh(x)
    return (1-t)*t

def relu(x):
    return max(0,x)

def relu_derivative(x):
    return 0 if x==0 else 1

def leaky_relu(x,alpha=0.01):
    return x if x>0 else alpha*x

def leaky_relu_derivative(x,alpha=0.01):
    return 1 if x>0 else alpha  


def softmax(xs):
    max_x = max(xs)
    exps = [math.exp(x-max_x) for x in xs]
    total = sum(exps)
    soft= [g/total for g in exps]
    return soft

In [6]:
def gradient_scan(name, derivative_fn, start=-5, end=5, n=100):
    step = (end - start) / n
    near_zero = 0
    healthy = 0
    for i in range(n):
        x = start + i * step
        g = derivative_fn(x)
        if abs(g) < 0.01:
            near_zero += 1
        else:
            healthy += 1
    pct_dead = near_zero / n * 100
    print(f"{name:15s}: {healthy:3d} healthy, {near_zero:3d} near-zero ({pct_dead:.0f}% dead zone)")

gradient_scan("Sigmoid", sigmoid_derivative)
gradient_scan("Tanh", tanh_derivative)
gradient_scan("ReLU", relu_derivative)
gradient_scan("Leaky ReLU", leaky_relu_derivative)
# gradient_scan("GELU", gelu_derivative)
# gradient_scan("Swish", swish_derivative)

Sigmoid        :  91 healthy,   9 near-zero (9% dead zone)
Tanh           :  76 healthy,  24 near-zero (24% dead zone)
ReLU           :  99 healthy,   1 near-zero (1% dead zone)
Leaky ReLU     : 100 healthy,   0 near-zero (0% dead zone)


In [8]:
import random

def vanishing_gradient_experiment(activation_fn, name, n_layers=10, n_inputs=5):
    random.seed(42)
    values = [random.gauss(0, 1) for _ in range(n_inputs)]

    print(f"\n{name} through {n_layers} layers:")
    for layer in range(n_layers):
        weights = [random.gauss(0, 1) for _ in range(n_inputs)]
        z = sum(w * v for w, v in zip(weights, values))
        activated = activation_fn(z)
        magnitude = abs(activated)
        bar = "#" * int(magnitude * 20)
        print(f"  Layer {layer+1:2d}: magnitude = {magnitude:.6f} {bar}")
        values = [activated] * n_inputs

vanishing_gradient_experiment(sigmoid, "Sigmoid")
vanishing_gradient_experiment(relu, "ReLU")
# vanishing_gradient_experiment(gelu, "GELU")


Sigmoid through 10 layers:
  Layer  1: magnitude = 0.505242 ##########
  Layer  2: magnitude = 0.672564 #############
  Layer  3: magnitude = 0.579702 ###########
  Layer  4: magnitude = 0.518791 ##########
  Layer  5: magnitude = 0.506911 ##########
  Layer  6: magnitude = 0.070461 #
  Layer  7: magnitude = 0.502930 ##########
  Layer  8: magnitude = 0.645463 ############
  Layer  9: magnitude = 0.521604 ##########
  Layer 10: magnitude = 0.662811 #############

ReLU through 10 layers:
  Layer  1: magnitude = 0.020967 
  Layer  2: magnitude = 0.029871 
  Layer  3: magnitude = 0.014281 
  Layer  4: magnitude = 0.001853 
  Layer  5: magnitude = 0.000099 
  Layer  6: magnitude = 0.000000 
  Layer  7: magnitude = 0.000000 
  Layer  8: magnitude = 0.000000 
  Layer  9: magnitude = 0.000000 
  Layer 10: magnitude = 0.000000 


In [9]:
def dead_neuron_detector(n_inputs=5, hidden_size=20, n_samples=1000):
    random.seed(0)
    weights = [[random.gauss(0, 1) for _ in range(n_inputs)] for _ in range(hidden_size)]
    biases = [random.gauss(0, 1) for _ in range(hidden_size)]

    fire_counts = [0] * hidden_size

    for _ in range(n_samples):
        inputs = [random.gauss(0, 1) for _ in range(n_inputs)]
        for neuron_idx in range(hidden_size):
            z = sum(w * x for w, x in zip(weights[neuron_idx], inputs)) + biases[neuron_idx]
            if relu(z) > 0:
                fire_counts[neuron_idx] += 1

    dead = sum(1 for c in fire_counts if c == 0)
    rarely_fire = sum(1 for c in fire_counts if 0 < c < n_samples * 0.05)
    healthy = hidden_size - dead - rarely_fire

    print(f"\nDead Neuron Report ({hidden_size} neurons, {n_samples} samples):")
    print(f"  Dead (never fired):     {dead}")
    print(f"  Barely alive (<5%):     {rarely_fire}")
    print(f"  Healthy:                {healthy}")
    print(f"  Dead neuron rate:       {dead/hidden_size*100:.1f}%")

    for i, c in enumerate(fire_counts):
        status = "DEAD" if c == 0 else "WEAK" if c < n_samples * 0.05 else "OK"
        bar = "#" * (c * 40 // n_samples)
        print(f"  Neuron {i:2d}: {c:4d}/{n_samples} fires [{status:4s}] {bar}")

dead_neuron_detector()


Dead Neuron Report (20 neurons, 1000 samples):
  Dead (never fired):     0
  Barely alive (<5%):     0
  Healthy:                20
  Dead neuron rate:       0.0%
  Neuron  0:  258/1000 fires [OK  ] ##########
  Neuron  1:  187/1000 fires [OK  ] #######
  Neuron  2:  500/1000 fires [OK  ] ####################
  Neuron  3:  654/1000 fires [OK  ] ##########################
  Neuron  4:  596/1000 fires [OK  ] #######################
  Neuron  5:  551/1000 fires [OK  ] ######################
  Neuron  6:  540/1000 fires [OK  ] #####################
  Neuron  7:  311/1000 fires [OK  ] ############
  Neuron  8:  545/1000 fires [OK  ] #####################
  Neuron  9:  454/1000 fires [OK  ] ##################
  Neuron 10:  615/1000 fires [OK  ] ########################
  Neuron 11:  750/1000 fires [OK  ] ##############################
  Neuron 12:  723/1000 fires [OK  ] ############################
  Neuron 13:  617/1000 fires [OK  ] ########################
  Neuron 14:  743/1000 fires [OK

In [11]:
def make_circle_data(n=200, seed=42):
    random.seed(seed)
    data = []
    for _ in range(n):
        x = random.uniform(-2, 2)
        y = random.uniform(-2, 2)
        label = 1.0 if x * x + y * y < 1.5 else 0.0
        data.append(([x, y], label))
    return data


class ActivationNetwork:
    def __init__(self, activation_fn, activation_deriv, hidden_size=8, lr=0.1):
        random.seed(0)
        self.act = activation_fn
        self.act_d = activation_deriv
        self.lr = lr
        self.hidden_size = hidden_size

        self.w1 = [[random.gauss(0, 0.5) for _ in range(2)] for _ in range(hidden_size)]
        self.b1 = [0.0] * hidden_size
        self.w2 = [random.gauss(0, 0.5) for _ in range(hidden_size)]
        self.b2 = 0.0

    def forward(self, x):
        self.x = x
        self.z1 = []
        self.h = []
        for i in range(self.hidden_size):
            z = self.w1[i][0] * x[0] + self.w1[i][1] * x[1] + self.b1[i]
            self.z1.append(z)
            self.h.append(self.act(z))

        self.z2 = sum(self.w2[i] * self.h[i] for i in range(self.hidden_size)) + self.b2
        self.out = sigmoid(self.z2)
        return self.out

    def backward(self, target):
        error = self.out - target
        d_out = error * self.out * (1 - self.out)

        for i in range(self.hidden_size):
            d_h = d_out * self.w2[i] * self.act_d(self.z1[i])
            self.w2[i] -= self.lr * d_out * self.h[i]
            for j in range(2):
                self.w1[i][j] -= self.lr * d_h * self.x[j]
            self.b1[i] -= self.lr * d_h
        self.b2 -= self.lr * d_out

    def train(self, data, epochs=200):
        losses = []
        for epoch in range(epochs):
            total_loss = 0
            correct = 0
            for x, y in data:
                pred = self.forward(x)
                self.backward(y)
                total_loss += (pred - y) ** 2
                if (pred >= 0.5) == (y >= 0.5):
                    correct += 1
            avg_loss = total_loss / len(data)
            accuracy = correct / len(data) * 100
            losses.append(avg_loss)
            if epoch % 50 == 0 or epoch == epochs - 1:
                print(f"    Epoch {epoch:3d}: loss={avg_loss:.4f}, accuracy={accuracy:.1f}%")
        return losses


data = make_circle_data()

configs = [
    ("Sigmoid", sigmoid, sigmoid_derivative),
    ("ReLU", relu, relu_derivative),
    # ("GELU", gelu, gelu_derivative),
]

results = {}
for name, act_fn, act_d_fn in configs:
    print(f"\n=== Training with {name} ===")
    net = ActivationNetwork(act_fn, act_d_fn, hidden_size=8, lr=0.1)
    losses = net.train(data, epochs=200)
    results[name] = losses

print("\n=== Final Loss Comparison ===")
for name, losses in results.items():
    print(f"  {name:10s}: start={losses[0]:.4f} -> end={losses[-1]:.4f} (improvement: {(1 - losses[-1]/losses[0])*100:.1f}%)")


=== Training with Sigmoid ===
    Epoch   0: loss=0.2222, accuracy=65.0%
    Epoch  50: loss=0.1693, accuracy=71.5%
    Epoch 100: loss=0.0609, accuracy=96.0%
    Epoch 150: loss=0.0413, accuracy=97.5%
    Epoch 199: loss=0.0319, accuracy=97.5%

=== Training with ReLU ===
    Epoch   0: loss=0.2430, accuracy=61.0%
    Epoch  50: loss=0.1309, accuracy=80.0%
    Epoch 100: loss=0.1325, accuracy=80.0%
    Epoch 150: loss=0.1337, accuracy=80.0%
    Epoch 199: loss=0.1352, accuracy=80.0%

=== Final Loss Comparison ===
  Sigmoid   : start=0.2222 -> end=0.0319 (improvement: 85.6%)
  ReLU      : start=0.2430 -> end=0.1352 (improvement: 44.4%)
